# 06 — Deep Learning Training

**Goal:** Train one deep learning model (GRU as a representative example) through the full training pipeline: learning-rate grid search, epoch-level training loop with weighted BCE loss, early stopping monitored by validation MCC, and loss / MCC visualisation.

**Modules used:** `src/training/trainer.py`, `src/training/losses.py`, `src/training/callbacks.py`

---

## 0 · Imports & data pipeline

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src.data.load_data import load_market_data
from src.data.preprocess import apply_missing_value_policy, select_columns
from src.data.labeling import compute_forward_return, compute_threshold, make_labels
from src.data.splitters import split_dev_test, make_expanding_folds
from src.data.sequence_builder import build_sequences, drop_neutral_sequences
from src.models.model_factory import build_model
from src.training.losses import build_bce_with_logits_loss
from src.training.callbacks import EarlyStopping
from src.training.trainer import (
    create_torch_dataloader,
    predict_proba_torch_model,
    evaluate_torch_model,
    run_single_torch_fold,
)
from src.evaluation.metrics import summarize_fold_metrics
from src.utils.seed import set_global_seed

set_global_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

DATA_PATH    = ROOT / 'data' / 'raw' / 'market_data_10y_enriched.csv'
FEATURE_COLS = ['VIX_Term_Structure','Yield_Curve','SKEW_Index',
                'Risk_Appetite_Ratio','Crude_Oil','VWAP_Deviation','Volume_Momentum']
DATE_COL   = 'Date'
CLOSE_COL  = 'Nasdaq_Close'
LOOKBACK   = 21
N_FOLDS    = 4
VAL_RATIO  = 0.10
TEST_RATIO = 0.15

In [ ]:
df_raw   = load_market_data(str(DATA_PATH), date_col=DATE_COL)
df_clean = apply_missing_value_policy(df_raw, method='ffill_then_drop_head')
df       = select_columns(df_clean, FEATURE_COLS, CLOSE_COL, DATE_COL)

dev_df, test_df = split_dev_test(df, test_ratio=TEST_RATIO)
folds = make_expanding_folds(len(dev_df), n_folds=N_FOLDS, val_ratio_within_dev=VAL_RATIO)
print(f'Dev: {len(dev_df)} | Test: {len(test_df)} | Folds: {len(folds)}')

---
## 1 · Prepare one fold for detailed walkthrough

We use **Fold 1** (smallest training set, fastest to run) to illustrate the full training process.

In [ ]:
tr_range, vl_range = folds[0]
train_df = dev_df.iloc[tr_range].reset_index(drop=True)
val_df   = dev_df.iloc[vl_range].reset_index(drop=True)

# Labeling
train_fwd = compute_forward_return(train_df[CLOSE_COL], horizon=1)
val_fwd   = compute_forward_return(val_df[CLOSE_COL],   horizon=1)
threshold = compute_threshold(train_fwd, method='quantile', quantile=0.40)
train_labels = make_labels(train_fwd, threshold)
val_labels   = make_labels(val_fwd,   threshold)

# Sequences
X_tr, y_tr, _ = build_sequences(train_df, FEATURE_COLS, train_labels, LOOKBACK)
X_vl, y_vl, _ = build_sequences(val_df,   FEATURE_COLS, val_labels,   LOOKBACK)
X_tr, y_tr, _ = drop_neutral_sequences(X_tr, y_tr, pd.Series(range(len(y_tr))))
X_vl, y_vl, _ = drop_neutral_sequences(X_vl, y_vl, pd.Series(range(len(y_vl))))

# Scaling (fit on train)
from sklearn.preprocessing import StandardScaler
n_tr, L, F = X_tr.shape
scaler = StandardScaler()
X_tr_2d = scaler.fit_transform(X_tr.reshape(-1, F))
X_vl_2d = scaler.transform(X_vl.reshape(-1, F))
X_tr = X_tr_2d.reshape(n_tr, L, F).astype(np.float32)
X_vl = X_vl_2d.reshape(len(X_vl), L, F).astype(np.float32)

print(f'Train: X={X_tr.shape}  y={y_tr.shape}  (Bull: {(y_tr==1).sum()}, Bear: {(y_tr==0).sum()})')
print(f'Val  : X={X_vl.shape}  y={y_vl.shape}')

---
## 2 · Loss function: Weighted Binary Cross-Entropy

Because Bull and Bear classes are not perfectly balanced, we weight the positive class:

$$\text{pos\_weight} = \frac{N_{\text{bear}}}{N_{\text{bull}}}$$

This tells the loss to penalise missing Bull signals more heavily than Bear signals, balancing effective learning.

In [ ]:
loss_fn = build_bce_with_logits_loss(y_tr, use_weighted_loss=True, device=DEVICE)
print(loss_fn)

n_bear = (y_tr == 0).sum()
n_bull = (y_tr == 1).sum()
print(f'Bear samples: {n_bear}  |  Bull samples: {n_bull}')
print(f'pos_weight = {n_bear / n_bull:.4f}')

---
## 3 · Early stopping

`EarlyStopping` monitors validation MCC each epoch. If the score does not improve for `patience=10` consecutive epochs, training halts and the best weights are restored.

In [ ]:
early_stopper = EarlyStopping(patience=10, higher_is_better=True)
print(f'Patience         : {early_stopper.patience}')
print(f'Monitor metric   : validation MCC (higher is better)')
print(f'Behavior on stop : restore best weights')

---
## 4 · Manual training loop (GRU — Fold 1)

This cell demonstrates **exactly what `run_single_torch_fold` does internally**, step by step.

In [ ]:
GRU_CONFIG = {
    'gru': {'hidden_dim': 64, 'num_layers': 1, 'dropout': 0.2, 'weighted_loss': True},
    'training': {'batch_size': 64, 'max_epochs': 60, 'learning_rate': 0.001,
                 'early_stopping_patience': 10},
}

INPUT_SHAPE = (LOOKBACK, len(FEATURE_COLS))
model = build_model('gru', GRU_CONFIG, input_shape=INPUT_SHAPE).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=GRU_CONFIG['training']['learning_rate'])
early_stop = EarlyStopping(patience=GRU_CONFIG['training']['early_stopping_patience'], higher_is_better=True)

train_loader = create_torch_dataloader(X_tr, y_tr, batch_size=64, shuffle=True)

train_losses, val_mccs = [], []
MAX_EPOCHS = GRU_CONFIG['training']['max_epochs']

for epoch in range(1, MAX_EPOCHS + 1):
    # --- Training step ---
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.float().to(DEVICE)

        optimizer.zero_grad()
        logits = model(X_batch).squeeze(1)
        loss   = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)

    # --- Validation step ---
    val_result = evaluate_torch_model(model, X_vl, y_vl, batch_size=64, device=DEVICE)
    val_mcc = val_result['mcc']
    val_mccs.append(val_mcc)

    improved = early_stop.step(val_mcc, model, epoch)
    marker = ' ◄ best' if improved else ''

    if epoch % 5 == 0 or improved:
        print(f'Epoch {epoch:3d}/{MAX_EPOCHS} | loss={avg_loss:.4f} | val_mcc={val_mcc:.4f}{marker}')

    if early_stop.stopped:
        print(f'\nEarly stopping triggered at epoch {epoch}. Best epoch: {early_stop.best_epoch}')
        early_stop.restore(model)
        break

print('\nTraining complete.')

---
## 5 · Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Loss curve
axes[0].plot(range(1, len(train_losses)+1), train_losses, color='steelblue', linewidth=1.2)
axes[0].set_title('Training Loss (BCE)', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

# MCC curve
best_epoch = early_stop.best_epoch
axes[1].plot(range(1, len(val_mccs)+1), val_mccs, color='mediumseagreen', linewidth=1.2)
axes[1].axvline(best_epoch, color='red', linestyle='--', linewidth=1, label=f'Best epoch ({best_epoch})')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle=':')
axes[1].set_title('Validation MCC', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MCC')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 6 · Full CV using `run_single_torch_fold`

Now we run all 4 folds using the high-level API that wraps the loop above, plus automatic LR search.

In [ ]:
FULL_CONFIG = {
    'gru': {'hidden_dim': 64, 'num_layers': 1, 'dropout': 0.2, 'weighted_loss': True},
    'training': {
        'batch_size': 64, 'max_epochs': 80,
        'early_stopping_patience': 10, 'learning_rate': 0.001,
        'monitor_metric': 'mcc',
    },
}

fold_metrics_dl = []

for fold_idx, (tr_range, vl_range) in enumerate(folds):
    train_df_f = dev_df.iloc[tr_range].reset_index(drop=True)
    val_df_f   = dev_df.iloc[vl_range].reset_index(drop=True)

    train_fwd_f = compute_forward_return(train_df_f[CLOSE_COL], horizon=1)
    val_fwd_f   = compute_forward_return(val_df_f[CLOSE_COL],   horizon=1)
    thr_f = compute_threshold(train_fwd_f, method='quantile', quantile=0.40)

    Xtr, ytr, _ = build_sequences(train_df_f, FEATURE_COLS,
                                  make_labels(train_fwd_f, thr_f), LOOKBACK)
    Xvl, yvl, _ = build_sequences(val_df_f, FEATURE_COLS,
                                  make_labels(val_fwd_f, thr_f), LOOKBACK)
    Xtr, ytr, _ = drop_neutral_sequences(Xtr, ytr, pd.Series(range(len(ytr))))
    Xvl, yvl, _ = drop_neutral_sequences(Xvl, yvl, pd.Series(range(len(yvl))))

    sc = StandardScaler()
    n, L, F = Xtr.shape
    Xtr = sc.fit_transform(Xtr.reshape(-1, F)).reshape(n, L, F).astype(np.float32)
    Xvl = sc.transform(Xvl.reshape(-1, F)).reshape(len(Xvl), L, F).astype(np.float32)

    set_global_seed(42)
    result = run_single_torch_fold(
        model_name='gru', config=FULL_CONFIG, input_shape=(LOOKBACK, F),
        X_train=Xtr, y_train=ytr, X_val=Xvl, y_val=yvl, device=DEVICE
    )
    fold_metrics_dl.append(result['val_metrics'])
    mcc = result['val_metrics']['mcc']
    f1  = result['val_metrics']['f1']
    print(f'Fold {fold_idx+1} | val_mcc={mcc:.4f}  val_f1={f1:.4f}')

summary_dl = summarize_fold_metrics(fold_metrics_dl)
print(f'\nGRU CV — MCC: {summary_dl["mcc_mean"]:.4f} ± {summary_dl["mcc_std"]:.4f}')
print(f'GRU CV — F1 : {summary_dl["f1_mean"]:.4f} ± {summary_dl["f1_std"]:.4f}')

---
## Summary

| Component | Implementation |
|---|---|
| Loss function | `BCEWithLogitsLoss` + `pos_weight` for class imbalance |
| Optimiser | Adam (`lr=0.001`) |
| Early stopping | Patience=10 epochs, monitor val MCC, restore best weights |
| CV | 4-fold expanding window, scaler fit on train only |

**Next:** `07_evaluation_and_comparison.ipynb` — confusion matrices, full metric tables, and best-model selection.